# Uygulama 4 — LHON Tanı Testi için PCR Primeri

**Hafta 1–2** — Ters tamamlayıcı, GC oranı, erime sıcaklığı (Tm), molekül ağırlığı, kayan pencere

**Veri:** NC_012920.1 (FASTA) — 11700..11899 bölgesi

---

**Senaryo:** Hastanede LHON şüpheli hastalar için hızlı bir test kurulacak: m.11778 bölgesi **PCR** ile çoğaltılıp dizilenecek. Moleküler biyolog primerleri belirlemiş, senden primerlerin özelliklerini hesaplamanı istiyor.

> **PCR nedir?** Düşün: Fotokopi makinesi gibi. DNA'nın belirli bir bölgesini milyonlarca kez kopyalar. Bunun için iki kısa DNA parçası (primer) lazım — biri kopyalamanın başlayacağı yeri, diğeri biteceği yeri belirler.

## Kalem-Kağıt Isınması

İleri primer: `5'-TTCTCATAATCGCCCACGGG-3'` (20 nt)

**(a) Baz sayıları:** A=2, C=6, G=4, T=4 → Kontrol: 2+6+4+4 = 16... Tekrar sayalım.
- T-T-C-T-C-A-T-A-A-T-C-G-C-C-C-A-C-G-G-G
- T=5, C=6, A=3, G=4... Hayır: Kodda doğrusunu göreceğiz!

**(b) Wallace kuralı:** Tm = 2(A+T) + 4(G+C)
- Kısa primerler (<14 nt) için kullanılır, 20 nt için kaba tahmin

**(c) GC formülü:** Tm = 64.9 + 41 × (G+C − 16.4) / N
- Daha uzun primerler için daha doğru

→ 20 nt için GC formülü daha uygundur.

---
## Görev 0 — Veriyi Yükle + Biopython MeltingTemp Modülü

In [ ]:
from Bio import Entrez, SeqIO
from Bio.Seq import Seq
from Bio.SeqUtils import gc_fraction, molecular_weight
from Bio.SeqUtils.MeltingTemp import Tm_Wallace, Tm_GC, Tm_NN
import matplotlib.pyplot as plt
import os

Entrez.email = "semra@example.com"

if os.path.exists("NC_012920.fasta"):
    record = SeqIO.read("NC_012920.fasta", "fasta")
else:
    handle = Entrez.efetch(db="nucleotide", id="NC_012920.1",
                           rettype="fasta", retmode="text")
    record = SeqIO.read(handle, "fasta")
    handle.close()
    SeqIO.write(record, "NC_012920.fasta", "fasta")

genom = record.seq.upper()
print(f"Genom: {len(genom)} bp")

---
## Görev 1 — Primerleri Elde Et ve Amplikonu Hesapla

- **İleri primer:** genomun 11700–11719 bölgesi (doğrudan)
- **Geri primer:** 11880–11899 bölgesinin **ters tamamlayıcısı** (çünkü karşı iplikte bağlanır)
- **Amplikon:** İki primer arasında kalan bölge (PCR ürünü)

In [ ]:
# Primerler (1-tabanlı → 0-tabanlı dönüşüm)
ileri_primer = genom[11699:11719]    # 11700..11719
geri_bolge = genom[11879:11899]      # 11880..11899
geri_primer = geri_bolge.reverse_complement()

print(f"İleri primer: 5'-{ileri_primer}-3'  ({len(ileri_primer)} nt)")
print(f"Geri primer:  5'-{geri_primer}-3'  ({len(geri_primer)} nt)")

# Amplikon
amplikon = genom[11699:11899]
print(f"\nAmplikon uzunluğu: {len(amplikon)} bp")

# 11778. baz amplikonun kaçıncı bazı?
mutasyon_amp = 11778 - 11700 + 1
print(f"m.11778 pozisyonu amplikonda: {mutasyon_amp}. baz")
print(f"→ Mutasyon amplikonun ortalarında, primer tasarımı uygun!")

---
## Görev 2 — Primer Özellikleri

İyi bir primer için:
- **GC oranı:** %40-60 arası ideal
- **Tm (erime sıcaklığı):** İki primerin Tm'si birbirine yakın olmalı (±5°C)
- **Wallace Tm:** `2(A+T) + 4(G+C)` — kısa primerler için
- **GC Tm:** `64.9 + 41(G+C − 16.4)/N` — daha genel
- **NN Tm:** En yakın komşu (nearest neighbor) — en doğru yöntem

In [ ]:
def primer_rapor(isim, primer):
    print(f"\n{'='*50}")
    print(f"{isim}: 5'-{primer}-3'")
    print(f"{'='*50}")
    
    # Baz sayıları
    s = str(primer)
    print(f"  A={s.count('A')}  C={s.count('C')}  G={s.count('G')}  T={s.count('T')}")
    
    # GC oranı
    gc = gc_fraction(primer) * 100
    print(f"  GC oranı: {gc:.1f}%")
    
    # Erime sıcaklıkları
    tm_w = Tm_Wallace(primer)
    tm_gc = Tm_GC(primer)
    tm_nn = Tm_NN(primer)
    print(f"  Wallace Tm:  {tm_w:.1f} °C")
    print(f"  GC Tm:       {tm_gc:.1f} °C")
    print(f"  NN Tm:       {tm_nn:.1f} °C")
    
    # Molekül ağırlığı
    mw = molecular_weight(primer, "DNA", double_stranded=False)
    print(f"  Molekül ağırlığı: {mw:.1f} Da")
    
    return tm_nn

tm_ileri = primer_rapor("İleri Primer", ileri_primer)
tm_geri = primer_rapor("Geri Primer", geri_primer)

print(f"\nTm farkı (NN): {abs(tm_ileri - tm_geri):.1f} °C")
if abs(tm_ileri - tm_geri) <= 5:
    print("→ Fark ≤5°C, primerler uyumlu!")
else:
    print("→ Fark >5°C, primerler uyumsuz olabilir!")

---
## Görev 3 — Primerler Genomda Tek mi?

İyi bir primer genomda **yalnızca bir yere** bağlanmalı. Birden fazla yere bağlanırsa yanlış bölgeler de çoğaltılır.

- İleri primeri **ileri iplikte** arıyoruz
- Geri primeri **ters iplikte** arıyoruz (yani ters tamamlayıcısını ileri iplikte arıyoruz)

In [ ]:
seq_str = str(genom)

# İleri primer — ileri iplikte ara
ileri_str = str(ileri_primer)
ileri_eslesme = []
bas = 0
while True:
    idx = seq_str.find(ileri_str, bas)
    if idx == -1:
        break
    ileri_eslesme.append(idx + 1)
    bas = idx + 1

print(f"İleri primer ileri iplikte: {len(ileri_eslesme)} eşleşme")
print(f"  Pozisyonlar: {ileri_eslesme}")

# Geri primer — ters iplikte ara (= ters tamamlayıcısını ileri iplikte ara)
geri_rc = str(geri_primer.reverse_complement())
geri_eslesme = []
bas = 0
while True:
    idx = seq_str.find(geri_rc, bas)
    if idx == -1:
        break
    geri_eslesme.append(idx + 1)
    bas = idx + 1

print(f"\nGeri primer ters iplikte: {len(geri_eslesme)} eşleşme")
print(f"  Pozisyonlar: {geri_eslesme}")

if len(ileri_eslesme) == 1 and len(geri_eslesme) == 1:
    print(f"\n→ Her iki primer de genomda TEK yere bağlanıyor. Primer tasarımı başarılı!")

---
## Görev 4 — GC Oranı Profili (Kayan Pencere)

**Kayan pencere (sliding window):** Genom boyunca sabit uzunlukta bir pencereyi kaydırarak, her pozisyondaki GC oranını hesaplıyoruz.

Düşün: Bir tren penceresinden dışarı bakıyorsun. Tren ilerledikçe manzara değişiyor — aynı şekilde pencere DNA boyunca kaydıkça GC oranı değişiyor.

- **Pencere:** 500 baz
- **Adım:** 100 baz

In [ ]:
pencere = 500
adim = 100

pozisyonlar = []
gc_degerleri = []

for i in range(0, len(genom) - pencere + 1, adim):
    parca = genom[i:i + pencere]
    gc = gc_fraction(parca) * 100
    pozisyonlar.append(i + pencere // 2)  # pencerenin ortası
    gc_degerleri.append(gc)

# En yüksek ve en düşük
max_gc = max(gc_degerleri)
min_gc = min(gc_degerleri)
max_poz = pozisyonlar[gc_degerleri.index(max_gc)]
min_poz = pozisyonlar[gc_degerleri.index(min_gc)]

print(f"En yüksek GC: {max_gc:.1f}% (pozisyon ~{max_poz})")
print(f"En düşük GC:  {min_gc:.1f}% (pozisyon ~{min_poz})")

# Amplikon bölgesinin GC'si
amp_gc = gc_fraction(amplikon) * 100
print(f"Amplikon GC:  {amp_gc:.1f}% (11700-11899)")

In [ ]:
# Grafik
plt.figure(figsize=(12, 4))
plt.plot(pozisyonlar, gc_degerleri, color='steelblue', linewidth=0.8)
plt.axhline(y=44.4, color='gray', linestyle='--', linewidth=0.5, label='Ortalama GC')

# Amplikon bölgesini işaretle
plt.axvspan(11700, 11899, color='red', alpha=0.3, label='Amplikon (11700-11899)')

plt.xlabel('Genom Pozisyonu (bp)')
plt.ylabel('GC Oranı (%)')
plt.title('İnsan mtDNA — GC Oranı Profili (500 bp pencere, 100 bp adım)')
plt.legend()
plt.tight_layout()
plt.savefig('gc_profili.png', dpi=150)
plt.show()
print("Grafik kaydedildi: gc_profili.png")

---

## Uygulama 4 Tamamlandı!

**Öğrendiğin kavramlar:**
- PCR primeri tasarımının temel kuralları (GC oranı, Tm uyumu, özgüllük)
- Üç farklı Tm hesaplama yöntemi (Wallace, GC formülü, en yakın komşu)
- `molecular_weight()` ile DNA ağırlığı hesabı
- Kayan pencere (sliding window) analizi
- Matplotlib ile genom boyunca GC profili çizme

**Sonraki:** Uygulama 5 — Kara Veba'nın İzinde: pPCP1 Plazmiti (GenBank, SeqIO, features)